# **Ajuste y Preprocesamiento de Datos**


In [1]:
import pyarrow as pa
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer


In [2]:

DATA_DIR = Path.cwd().parent / "data" / "processed"
print(DATA_DIR)

c:\Users\belac\OneDrive\Documentos\Universidad\Ciencia de Datos y ML\Mlops\credit-risk-classification\data\processed


In [3]:
datos_cooperativa = pd.read_parquet(DATA_DIR / "01_datos_crudos_cooperativa.parquet")

In [4]:
datos_cooperativa.info()

<class 'pandas.DataFrame'>
RangeIndex: 12910 entries, 0 to 12909
Data columns (total 28 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   cartera             12910 non-null  str    
 1   plazo               12910 non-null  int64  
 2   vinculacion         12910 non-null  int64  
 3   valor_cuota         12910 non-null  float64
 4   valor_prestamo      12910 non-null  float64
 5   saldo_capital       12910 non-null  float64
 6   saldo_interes       12910 non-null  int64  
 7   aportes             12910 non-null  int64  
 8   garantias           12910 non-null  int64  
 9   valorgarantia       12910 non-null  int64  
 10  reestr              12910 non-null  int64  
 11  ctasahorros         12910 non-null  float64
 12  edad                12910 non-null  float64
 13  tipoasociado        12910 non-null  int64  
 14  estado_cliente      12910 non-null  int64  
 15  departamento        12909 non-null  str    
 16  ciudad         

# **Eliminación de Columnas**

La columna `ciudad` fue eliminada debido a que contiene una gran cantidad de categorías diferentes, lo que puede generar alta cardinalidad en los datos. 

Esto incrementa la complejidad del preprocesamiento y puede afectar el rendimiento de algunos modelos de machine learning, especialmente aquellos que requieren codificación de variables categóricas. Además, la información geográfica relevante ya se encuentra representada de forma más general en otras variables como `departamento` o `grupo_ciudad`.


In [5]:
datos_cooperativa = datos_cooperativa.drop(columns=["ciudad"])

print(datos_cooperativa.columns)

Index(['cartera', 'plazo', 'vinculacion', 'valor_cuota', 'valor_prestamo',
       'saldo_capital', 'saldo_interes', 'aportes', 'garantias',
       'valorgarantia', 'reestr', 'ctasahorros', 'edad', 'tipoasociado',
       'estado_cliente', 'departamento', 'sexo', 'curtotalingresos',
       'curtotalegresos', 'intestrato', 'actualizacion', 'default',
       'puntaje_data', 'grupo_dptmto', 'grupo_ciudad', 'grupo_edad',
       'grupo_actividadeco'],
      dtype='str')


## **Identificación de Valores Faltantes (Nan, Null, None)**

In [6]:
datos_cooperativa.sample(8, random_state=1000)

,cartera,plazo,vinculacion,valor_cuota,valor_prestamo,saldo_capital,saldo_interes,aportes,garantias,valorgarantia,...,curtotalingresos,curtotalegresos,intestrato,actualizacion,default,puntaje_data,grupo_dptmto,grupo_ciudad,grupo_edad,grupo_actividadeco
9780,consumo_sin_libranza,1096,827,166485.0,5100000.0,1551461.0,3720,304582,1,304582,...,1000000.0,200000.0,2.0,0,1,737.0,3,4,1,4
6445,consumo_sin_libranza,731,868,139299.0,2680000.0,1986847.0,127680,449372,1,449372,...,1300000.0,390000.0,3.0,1,1,693.0,3,7,1,4
8716,consumo_con_libranza,2192,3481,883296.0,36000000.0,21353348.0,0,5523320,1,5523320,...,4750000.0,1000000.0,2.0,0,0,834.0,3,7,2,4
3535,consumo_sin_libranza,2557,2419,3122994.0,136900000.0,122644820.0,2073568,5239946,2,197271672,...,13300000.0,4000000.0,6.0,0,0,787.0,3,6,3,4
8902,consumo_sin_libranza,1826,1706,747096.0,35000000.0,30328543.0,0,2102369,1,2102369,...,6000000.0,500000.0,2.0,0,1,748.0,3,2,1,4
8229,consumo_sin_libranza,1461,1288,306272.0,10100000.0,1735917.0,1910,363050,1,2506689,...,4972000.0,800000.0,2.0,1,0,775.0,3,6,1,4
8934,consumo_sin_libranza,1096,1939,212608.0,5930000.0,3043287.0,645077,429711,1,429711,...,1027739.0,150000.0,3.0,0,1,772.0,3,7,1,4
4616,consumo_sin_libranza,2191,2165,593968.0,21950000.0,21332507.0,373325,1886050,1,1886050,...,2020776.0,500000.0,2.0,1,0,771.0,3,7,2,4


In [7]:
datos_cooperativa.isnull().sum()

cartera               0
plazo                 0
vinculacion           0
valor_cuota           0
valor_prestamo        0
saldo_capital         0
saldo_interes         0
aportes               0
garantias             0
valorgarantia         0
reestr                0
ctasahorros           0
edad                  0
tipoasociado          0
estado_cliente        0
departamento          1
sexo                  0
curtotalingresos      0
curtotalegresos       0
intestrato            0
actualizacion         0
default               0
puntaje_data          0
grupo_dptmto          0
grupo_ciudad          0
grupo_edad            0
grupo_actividadeco    0
dtype: int64

Eliminamos los valores nulos encontrados en las variables que en este caso hay simplemente un valor nulo en la variable `departamento`

In [8]:
datos_cooperativa = datos_cooperativa.dropna(subset=['departamento'])
datos_cooperativa.isnull().sum()

cartera               0
plazo                 0
vinculacion           0
valor_cuota           0
valor_prestamo        0
saldo_capital         0
saldo_interes         0
aportes               0
garantias             0
valorgarantia         0
reestr                0
ctasahorros           0
edad                  0
tipoasociado          0
estado_cliente        0
departamento          0
sexo                  0
curtotalingresos      0
curtotalegresos       0
intestrato            0
actualizacion         0
default               0
puntaje_data          0
grupo_dptmto          0
grupo_ciudad          0
grupo_edad            0
grupo_actividadeco    0
dtype: int64

Verificación final de todo el conjunto de datos después de la eliminación de los datos nulos encontrados.

In [9]:
datos_cooperativa.info()

<class 'pandas.DataFrame'>
Index: 12909 entries, 0 to 12909
Data columns (total 27 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   cartera             12909 non-null  str    
 1   plazo               12909 non-null  int64  
 2   vinculacion         12909 non-null  int64  
 3   valor_cuota         12909 non-null  float64
 4   valor_prestamo      12909 non-null  float64
 5   saldo_capital       12909 non-null  float64
 6   saldo_interes       12909 non-null  int64  
 7   aportes             12909 non-null  int64  
 8   garantias           12909 non-null  int64  
 9   valorgarantia       12909 non-null  int64  
 10  reestr              12909 non-null  int64  
 11  ctasahorros         12909 non-null  float64
 12  edad                12909 non-null  float64
 13  tipoasociado        12909 non-null  int64  
 14  estado_cliente      12909 non-null  int64  
 15  departamento        12909 non-null  str    
 16  sexo                

## **Conversión de Datos**

Las variables categoricas se encuentran en tipo **string**, por lo que vamos a conventirlas a tipo categoricas porque los modelos no trabajan directamente con texto

In [10]:
print(datos_cooperativa.select_dtypes(include=['object', 'string']))


                    cartera departamento
0      consumo_sin_libranza    antioquia
1      consumo_sin_libranza    antioquia
2      consumo_sin_libranza    antioquia
3      consumo_sin_libranza    antioquia
4      consumo_sin_libranza    antioquia
...                     ...          ...
12905  consumo_con_libranza    antioquia
12906  consumo_con_libranza    antioquia
12907  consumo_con_libranza    antioquia
12908  consumo_con_libranza    antioquia
12909  consumo_sin_libranza    antioquia

[12909 rows x 2 columns]


In [11]:
cols_categoricas = ['cartera', 'departamento']
datos_cooperativa[cols_categoricas] = datos_cooperativa[cols_categoricas].astype('category')

Verificamos que su tipo de dato se encuentre en el dataset

In [12]:
datos_cooperativa.info()

<class 'pandas.DataFrame'>
Index: 12909 entries, 0 to 12909
Data columns (total 27 columns):
 #   Column              Non-Null Count  Dtype   
---  ------              --------------  -----   
 0   cartera             12909 non-null  category
 1   plazo               12909 non-null  int64   
 2   vinculacion         12909 non-null  int64   
 3   valor_cuota         12909 non-null  float64 
 4   valor_prestamo      12909 non-null  float64 
 5   saldo_capital       12909 non-null  float64 
 6   saldo_interes       12909 non-null  int64   
 7   aportes             12909 non-null  int64   
 8   garantias           12909 non-null  int64   
 9   valorgarantia       12909 non-null  int64   
 10  reestr              12909 non-null  int64   
 11  ctasahorros         12909 non-null  float64 
 12  edad                12909 non-null  float64 
 13  tipoasociado        12909 non-null  int64   
 14  estado_cliente      12909 non-null  int64   
 15  departamento        12909 non-null  category
 16  se

## **Variables Categóricas**

In [13]:
datos_cooperativa[['cartera', 'departamento']]

,cartera,departamento
0,consumo_sin_libranza,antioquia
1,consumo_sin_libranza,antioquia
2,consumo_sin_libranza,antioquia
3,consumo_sin_libranza,antioquia
4,consumo_sin_libranza,antioquia
...,...,...
12905,consumo_con_libranza,antioquia
12906,consumo_con_libranza,antioquia
12907,consumo_con_libranza,antioquia
12908,consumo_con_libranza,antioquia


Nuestro dataset no presenta variables categóricas ordinales, entonces solo vamos a preparar variables categóricas nominales

In [14]:
# Visualizamos las categorias que tiene cada pregunta 

print(datos_cooperativa["cartera"].unique())
print(datos_cooperativa["departamento"].unique())

['consumo_sin_libranza', 'vivienda', 'consumo_con_libranza']
Categories (3, str): ['consumo_con_libranza', 'consumo_sin_libranza', 'vivienda']
['antioquia', 'valle_del_cauca', 'quindio', 'bogotá', 'nariño', ..., 'meta', 'norte_de_santander', 'magdalena', 'sucre', 'la_guajira']
Length: 23
Categories (23, str): ['antioquia', 'arauca', 'atlántico', 'bogotá', ..., 'santander', 'sucre', 'tolima', 'valle_del_cauca']


Para estas variables vamos a usar el método **One Hot Encoding** con pipelines para transformar los datos.

## **Variables Númericas**

In [15]:
datos_numericos = datos_cooperativa.select_dtypes(include=['int64'])
print(datos_numericos.columns)

Index(['plazo', 'vinculacion', 'saldo_interes', 'aportes', 'garantias',
       'valorgarantia', 'reestr', 'tipoasociado', 'estado_cliente', 'sexo',
       'actualizacion', 'default', 'grupo_dptmto', 'grupo_ciudad',
       'grupo_edad', 'grupo_actividadeco'],
      dtype='str')


En este caso las variables `valorgarantia` y `saldo_interes` las vamos a convertir a tipo de dato flotante ya que representan valores monetarios reales, por lo tanto pueden llegar a contener decimales.

In [22]:
datos_cooperativa["saldo_interes"] = datos_cooperativa["saldo_interes"].astype(float)
datos_cooperativa["valorgarantia"] = datos_cooperativa["valorgarantia"].astype(float)

In [23]:
datos_flotantes = datos_cooperativa.select_dtypes(include=['float64'])
print(datos_flotantes.columns)

Index(['valor_cuota', 'valor_prestamo', 'saldo_capital', 'saldo_interes',
       'valorgarantia', 'ctasahorros', 'edad', 'curtotalingresos',
       'curtotalegresos', 'intestrato', 'puntaje_data'],
      dtype='str')


In [18]:
datos_numericos = datos_cooperativa.select_dtypes(include=['int64'])
print(datos_numericos.columns)

Index(['plazo', 'vinculacion', 'aportes', 'garantias', 'reestr',
       'tipoasociado', 'estado_cliente', 'sexo', 'actualizacion', 'default',
       'grupo_dptmto', 'grupo_ciudad', 'grupo_edad', 'grupo_actividadeco'],
      dtype='str')


In [24]:
#Identificamos las columnas para crear el Pipeline de preprocesamiento

cols_numericas_enteras = ['plazo', 'vinculacion', 'aportes', 'garantias', 'reestr',
       'tipoasociado', 'estado_cliente', 'sexo', 'actualizacion',
       'grupo_dptmto', 'grupo_ciudad', 'grupo_edad', 'grupo_actividadeco']

cols_numericas_flotantes = ['valor_cuota', 'valor_prestamo', 'saldo_capital', 'saldo_interes',
       'valorgarantia', 'ctasahorros', 'edad', 'curtotalingresos',
       'curtotalegresos', 'intestrato', 'puntaje_data']

In [20]:
datos_cooperativa.info()

<class 'pandas.DataFrame'>
Index: 12909 entries, 0 to 12909
Data columns (total 27 columns):
 #   Column              Non-Null Count  Dtype   
---  ------              --------------  -----   
 0   cartera             12909 non-null  category
 1   plazo               12909 non-null  int64   
 2   vinculacion         12909 non-null  int64   
 3   valor_cuota         12909 non-null  float64 
 4   valor_prestamo      12909 non-null  float64 
 5   saldo_capital       12909 non-null  float64 
 6   saldo_interes       12909 non-null  float64 
 7   aportes             12909 non-null  int64   
 8   garantias           12909 non-null  int64   
 9   valorgarantia       12909 non-null  float64 
 10  reestr              12909 non-null  int64   
 11  ctasahorros         12909 non-null  float64 
 12  edad                12909 non-null  float64 
 13  tipoasociado        12909 non-null  int64   
 14  estado_cliente      12909 non-null  int64   
 15  departamento        12909 non-null  category
 16  se

Como vamos a trabajar con un modelo de Regresión Logística es recomendable estandarizar las númericas, porque este modelo se puede ver afectado por la escala de los datos. 

Para este paso vamos a guardar el dataset ya ajustado a sus tipos de datos para realizar el EDA

In [25]:
print(Path.cwd())

c:\Users\belac\OneDrive\Documentos\Universidad\Ciencia de Datos y ML\Mlops\credit-risk-classification\notebooks


In [ ]:

BASE_DIR = Path.cwd().parent

ruta_datos_procesados = BASE_DIR / "data" / "processed"

ruta_datos_procesados.mkdir(parents=True, exist_ok=True)

ruta_archivo = ruta_datos_procesados / "02_datos_ajustados_cooperativa.parquet"

datos_cooperativa.to_parquet(ruta_archivo, index=False)

In [27]:
datos_cooperativa_ajustados = pd.read_parquet(DATA_DIR / "02_datos_ajustados_cooperativa.parquet")

Verificamos que los datos ajustados para realizar el EDA se encuentren correctos

In [29]:
datos_cooperativa_ajustados.info()

<class 'pandas.DataFrame'>
RangeIndex: 12909 entries, 0 to 12908
Data columns (total 27 columns):
 #   Column              Non-Null Count  Dtype   
---  ------              --------------  -----   
 0   cartera             12909 non-null  category
 1   plazo               12909 non-null  int64   
 2   vinculacion         12909 non-null  int64   
 3   valor_cuota         12909 non-null  float64 
 4   valor_prestamo      12909 non-null  float64 
 5   saldo_capital       12909 non-null  float64 
 6   saldo_interes       12909 non-null  float64 
 7   aportes             12909 non-null  int64   
 8   garantias           12909 non-null  int64   
 9   valorgarantia       12909 non-null  float64 
 10  reestr              12909 non-null  int64   
 11  ctasahorros         12909 non-null  float64 
 12  edad                12909 non-null  float64 
 13  tipoasociado        12909 non-null  int64   
 14  estado_cliente      12909 non-null  int64   
 15  departamento        12909 non-null  category
 1

## **Creación de pipelines**

In [30]:
categorical_pipeline = Pipeline(steps=[
    ('encoder', OneHotEncoder(handle_unknown='ignore'))      
])

categorical_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('encoder', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"categories categories: 'auto' or a list of array-like, default='auto'Categories (unique values) per feature:- 'auto' : Determine categories automatically from the training data.- list : ``categories[i]`` holds the categories expected in the ith column. The passed categories should not mix strings and numeric values within a single feature, and should be sorted in case of numeric values.The used categories can be found in the ``categories_`` attribute... versionadded:: 0.20",'auto'
,"drop drop: {'first', 'if_binary'} or an array-like of shape (n_features,), default=NoneSpecifies a methodology to use to drop one of the categories perfeature. This is useful in situations where perfectly collinearfeatures cause problems, such as when feeding the resulting datainto an unregularized linear regression model.However, dropping one category breaks the symmetry of the originalrepresentation and can therefore induce a bias in downstream models,for instance for penalized linear classification or regression models.- None : retain all features (the default).- 'first' : drop the first category in each feature. If only one category is present, the feature will be dropped entirely.- 'if_binary' : drop the first category in each feature with two categories. Features with 1 or more than 2 categories are left intact.- array : ``drop[i]`` is the category in feature ``X[:, i]`` that should be dropped.When `max_categories` or `min_frequency` is configured to groupinfrequent categories, the dropping behavior is handled after thegrouping... versionadded:: 0.21 The parameter `drop` was added in 0.21... versionchanged:: 0.23 The option `drop='if_binary'` was added in 0.23... versionchanged:: 1.1 Support for dropping infrequent categories.",None
,"sparse_output sparse_output: bool, default=TrueWhen ``True``, it returns a :class:`scipy.sparse.csr_matrix`,i.e. a sparse matrix in ""Compressed Sparse Row"" (CSR) format... versionadded:: 1.2 `sparse` was renamed to `sparse_output`",True
,"dtype dtype: number type, default=np.float64Desired dtype of output.",<class 'numpy.float64'>
,"handle_unknown handle_unknown: {'error', 'ignore', 'infrequent_if_exist', 'warn'}, default='error'Specif

In [31]:
numerical_pipeline = Pipeline(steps=[
    ('scaler', StandardScaler())            
])

numerical_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('scaler', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True


## **Creación ColumnTransformer**

In [33]:
preprocessor = ColumnTransformer(
    transformers=[
        ('numericas', numerical_pipeline, cols_numericas_enteras + cols_numericas_flotantes),
        ('categoricas_nominales', categorical_pipeline, cols_categoricas),
    ]
)

preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numericas', ...), ('categoricas_nominales', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and 

## **Train y Test**

In [34]:
X = datos_cooperativa_ajustados.drop(columns='default')
Y = datos_cooperativa_ajustados['default']


Dividimos los datos en 80% de entrenamiento y en 20% para Prueba

In [35]:
# 80% para train y 20% para test
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=100
)

Verificamos las dimensiones

In [36]:
X_train.shape, X_test.shape, Y_train.shape, Y_test.shape

((10327, 26), (2582, 26), (10327,), (2582,))

## **Transformación**

In [37]:
datos_cooperativa_transformados = preprocessor.fit_transform(X_train)
print(datos_cooperativa_transformados.shape)

(10327, 49)


In [38]:
datos_cooperativa_transformados

array([[-0.50395158,  0.62813238,  0.00923967, ...,  0.        ,
         0.        ,  0.        ],
       [ 0.06657882,  0.72629516,  0.92093972, ...,  0.        ,
         0.        ,  0.        ],
       [ 0.83249634, -0.18448871, -0.65029406, ...,  0.        ,
         0.        ,  0.        ],
       ...,
       [-0.20227386,  0.22529458, -0.43838751, ...,  0.        ,
         0.        ,  0.        ],
       [-1.07604507, -0.54889487, -0.5087906 , ...,  0.        ,
         0.        ,  0.        ],
       [ 1.20920271, -0.11549695,  0.06922939, ...,  0.        ,
         0.        ,  0.        ]], shape=(10327, 49))

In [39]:
feature_names = preprocessor.get_feature_names_out()

feature_names

array(['numericas__plazo', 'numericas__vinculacion', 'numericas__aportes',
       'numericas__garantias', 'numericas__reestr',
       'numericas__tipoasociado', 'numericas__estado_cliente',
       'numericas__sexo', 'numericas__actualizacion',
       'numericas__grupo_dptmto', 'numericas__grupo_ciudad',
       'numericas__grupo_edad', 'numericas__grupo_actividadeco',
       'numericas__valor_cuota', 'numericas__valor_prestamo',
       'numericas__saldo_capital', 'numericas__saldo_interes',
       'numericas__valorgarantia', 'numericas__ctasahorros',
       'numericas__edad', 'numericas__curtotalingresos',
       'numericas__curtotalegresos', 'numericas__intestrato',
       'numericas__puntaje_data',
       'categoricas_nominales__cartera_consumo_con_libranza',
       'categoricas_nominales__cartera_consumo_sin_libranza',
       'categoricas_nominales__cartera_vivienda',
       'categoricas_nominales__departamento_antioquia',
       'categoricas_nominales__departamento_atlántico',
     

In [40]:
datos_cooperativa_x = pd.DataFrame(
    datos_cooperativa_transformados,
    columns=feature_names
)

datos_cooperativa_x.head()

,numericas__plazo,numericas__vinculacion,numericas__aportes,numericas__garantias,numericas__reestr,numericas__tipoasociado,numericas__estado_cliente,numericas__sexo,numericas__actualizacion,numericas__grupo_dptmto,...,categoricas_nominales__departamento_magdalena,categoricas_nominales__departamento_meta,categoricas_nominales__departamento_nariño,categoricas_nominales__departamento_norte_de_santander,categoricas_nominales__departamento_quindio,categoricas_nominales__departamento_risaralda,categoricas_nominales__departamento_santander,categoricas_nominales__departamento_sucre,categoricas_nominales__departamento_tolima,categoricas_nominales__departamento_valle_del_cauca
0,-0.503952,0.628132,0.009240,-0.325871,0.009841,-1.110997,-0.080838,1.072434,-1.081030,0.12232,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.066579,0.726295,0.920940,-0.325871,0.009841,0.900093,-0.080838,1.072434,-1.081030,0.12232,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.832496,-0.184489,-0.650294,-0.325871,0.009841,0.900093,-0.080838,1.072434,0.925043,0.12232,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.066579,1.342591,0.731995,-0.325871,0.009841,-1.110997,-0.080838,-0.932458,0.925043,0.12232,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.637109,-0.123832,-0.074108,-0.325871,0.009841,0.900093,-0.080838,1.072434,-1.081030,0.12232,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
